### Gathering Historical Fantasy Scores

In the cells below I will be gathering historical fantasy football players points for a specified season. I will first look at the 2025 season and perform transformations, cleaning, and EDA to determine if more data is needed and identify holes. This data is useful because we can use the historical points as a way to compare out models output performance as in the accuracy of our output.

This data is gathered from ESPN's default-leage endpoint api: [Fantasy API 2025](https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/seasons/2025)

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from pathlib import Path
import os
#root directory of project
root = Path.cwd().parent
data_path = root/"data"

In [3]:
## load fantasy historical data
weekly_df = pd.read_csv(data_path/"player_weeks_2025.csv")
weekly_df.head()


,player_id,player,position,pro_team_id,week,fantasy_points,projected_points
0,4429795,Jahmyr Gibbs,RB,8,1,15.0,18.419261
1,3117251,Christian McCaffrey,RB,25,1,23.2,17.690153
2,4362628,Ja'Marr Chase,WR,4,1,4.6,19.783577
3,4426515,Puka Nacua,WR,14,1,23.1,16.765436
4,4430807,Bijan Robinson,RB,1,1,24.4,19.748249


In [4]:
# load player summary data
player_sum = pd.read_csv(data_path/"player_summary_2025.csv")
player_sum.head()

,player_id,player,position,team,games,avg_opp_def_rank,total_rushingAttempts,total_rushingYards,total_rushingTouchdowns,total_receptions,...,avg_fieldGoalPct,avg_fieldGoalsMadeYardsAverage,avg_avgInterceptionYards,avg_fantasy_points,longRushing,longReception,longPassing,longFieldGoalMade,longInterception,total_fantasy_points
0,3117251,Christian McCaffrey,RB,SF,17,16.000000,311.0,1202.0,10.0,102.0,...,NaN,NaN,NaN,24.505882,41.0,39.0,NaN,NaN,NaN,416.60
1,4426515,Puka Nacua,WR,LAR,16,15.750000,10.0,105.0,1.0,129.0,...,NaN,NaN,NaN,23.437500,45.0,58.0,NaN,NaN,NaN,375.00
2,4430807,Bijan Robinson,RB,ATL,17,16.529412,287.0,1478.0,7.0,79.0,...,NaN,NaN,NaN,21.811765,93.0,69.0,NaN,NaN,NaN,370.80
3,4429795,Jahmyr Gibbs,RB,DET,17,16.882353,243.0,1223.0,13.0,77.0,...,NaN,NaN,NaN,21.582353,78.0,42.0,NaN,NaN,NaN,366.90
4,3918298,Josh Allen,QB,BUF,17,16.470588,112.0,579.0,14.0,NaN,...,NaN,NaN,NaN,21.448235,40.0,NaN,54.0,NaN,NaN,364.62


In [5]:
# load player log data
player_logs = pd.read_csv(data_path/"player_gamelogs_2025.csv")
player_logs.head()


,player_id,player,position,team,season,season_type,week,eventId,date,opponent,...,fieldGoalsMade40_49,fieldGoalAttempts40_49,fieldGoalsMade50,fieldGoalAttempts50,fieldGoalsMade,fieldGoalAttempts,extraPointsMade,extraPointAttempts,opp_def_rank,fantasy_points
0,4429795,Jahmyr Gibbs,RB,DET,2025,2025 Regular Season,18,401772965,2026-01-04T21:25:00.000+00:00,CHI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23,20.3
1,4429795,Jahmyr Gibbs,RB,DET,2025,2025 Regular Season,17,401772711,2025-12-25T21:30:00.000+00:00,MIN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,6.4
2,4429795,Jahmyr Gibbs,RB,DET,2025,2025 Regular Season,16,401772806,2025-12-21T21:25:00.000+00:00,PIT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17,22.8
3,4429795,Jahmyr Gibbs,RB,DET,2025,2025 Regular Season,15,401772909,2025-12-14T21:25:00.000+00:00,LAR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10,9.8
4,4429795,Jahmyr Gibbs,RB,DET,2025,2025 Regular Season,14,401772947,2025-12-05T01:15:00.000+00:00,DAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32,37.0


In [6]:
display(weekly_df.shape)
display(player_sum.shape)
display(player_logs.shape)

(13958, 7)

(685, 104)

(7572, 70)

In [7]:
display(weekly_df["player_id"].nunique())
display(player_sum["player_id"].nunique())
display(player_logs["player_id"].nunique())

1088

685

685

In [23]:
## match our historical fantasy players with our other data
players_in_sum = player_sum["player_id"].unique()
players_set = set(players_in_sum)

players_in_logs = player_logs["player_id"].unique()
players_set_2 = set(players_in_logs)

assert players_set == players_set_2, "not same players"

In [24]:
# now that two of our df contain the same players, we only need to filter the historical fantasy data to contain the player_id as the other dataframes

weekly_df = weekly_df[weekly_df["player_id"].isin(players_set)]
weekly_df.shape

(11356, 7)

Displaying the Nan values gives us some information as to what is missing. We can see that the only columns with missing data are the fantasy_points column, and the projected_points column. These missing points in both indicate that the palyer didnt play that week and since we are interested in projected points, I will add a column which flags a player if they didnt play in a given week

In [8]:
mask = weekly_df[weekly_df["projected_points"].isna() & weekly_df["fantasy_points"].isna()]

# check why there are no columns with both no point and no projected points
no_proj = weekly_df[weekly_df["projected_points"].isna()]
no_pts = weekly_df[weekly_df["fantasy_points"].isna()]

display(mask.head())
display(no_proj.head())
display(no_pts.head())

print("length of no projected points:", len(no_proj))
print("length of no proj with week = 1:", len(no_proj[no_proj["week"] == 1]))
print(len(no_pts[no_pts["projected_points"] == 0]))

,player_id,player,position,pro_team_id,week,fantasy_points,projected_points


,player_id,player,position,pro_team_id,week,fantasy_points,projected_points
355,4601080,Raheim Sanders,RB,5,1,6.3,NaN
422,4430656,Myles Price,WR,16,1,0.0,NaN
450,4715355,Darius Cooper,WR,21,1,0.0,NaN
453,5160110,Jahdae Walker,WR,3,1,0.0,NaN
742,4876006,Zaire Mitchell-Paden,TE,18,1,0.0,NaN


,player_id,player,position,pro_team_id,week,fantasy_points,projected_points
144,4240657,Michael Carter,RB,22,1,NaN,0.0
149,4574716,Harrison Mevis,K,14,1,NaN,0.0
150,4430968,Kimani Vidal,RB,24,1,NaN,0.0
167,4034949,Eddy Pineiro,K,25,1,NaN,0.0
227,4569682,Audric Estime,RB,18,1,NaN,0.0


length of no projected points: 1202
length of no proj with week = 1: 9
2101


We can see that there are actually no rows where there are not fantasy points and no projected points, so i displayed both tables to see what is going on. We can see that if there are no fantasy points, then the projected points are 0 instead of nan. And we see there are some cases where there are fantasy points, but no projections. For the case where the fantasy points are NaN and the projected points are 0, I will remove these rows since they are of no use. For the rows which there are fantasy points and no projection, I will impute these values.